In [1]:
# ============================================================
# STAGE 2
# Student-Independent Sign Language Recognition
# 130 Features × Middle 80 Frames (Extracted from 120 Frames)
# ============================================================

import os
import time
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from collections import Counter

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_recall_fscore_support
)

import tensorflow as tf
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import (
    Input,
    Conv1D,
    LSTM,
    Dense,
    Dropout,
    BatchNormalization,
    Bidirectional,
    Attention,
    GlobalAveragePooling1D
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau
)


# ============================================================
# 1. SETTINGS
# ============================================================

BASE_DIR = r"D:\aaa EAAI Major Revision\dynamic\dynamic_dataset"

DATA_DIR = os.path.join(BASE_DIR, "csv_dataset")

TRAIN_PATH = os.path.join(DATA_DIR, "train_csv")
VAL_PATH = os.path.join(DATA_DIR, "validation_csv")
TEST_PATH = os.path.join(DATA_DIR, "test_csv")

# Updated results directory to reflect the frame change
RESULTS_DIR = os.path.join(
    BASE_DIR,
    "results_130features_middle80frames"
)

os.makedirs(RESULTS_DIR, exist_ok=True)

ORIGINAL_SEQUENCE_LENGTH = 120
TARGET_FRAMES = 60
NUM_FEATURES = 130
EXPECTED_CLASSES = 40

BATCH_SIZE = 64
EPOCHS = 200
LEARNING_RATE = 0.0005

np.random.seed(42)
tf.random.set_seed(42)


# ============================================================
# 2. CHECK DATASET STRUCTURE
# ============================================================

print("=" * 70)
print("CHECKING DATASET")
print("=" * 70)

print("Train path:", TRAIN_PATH)
print("Validation path:", VAL_PATH)
print("Test path:", TEST_PATH)

if not os.path.exists(TRAIN_PATH):
    raise FileNotFoundError(f"Train path not found: {TRAIN_PATH}")

if not os.path.exists(VAL_PATH):
    raise FileNotFoundError(f"Validation path not found: {VAL_PATH}")

if not os.path.exists(TEST_PATH):
    raise FileNotFoundError(f"Test path not found: {TEST_PATH}")


# ============================================================
# 3. LOAD SEQUENCES WITH MIDDLE SLICING
# ============================================================

def load_and_slice_sequences_from_folder(base_path):
    """
    Loads 120 frames per video and extracts the middle TARGET_FRAMES (e.g., 80).
    """
    class_names = sorted([
        d for d in os.listdir(base_path)
        if os.path.isdir(os.path.join(base_path, d))
    ])

    sequences = []
    labels = []

    print("\nLoading and slicing:", base_path)
    print("Classes found:", len(class_names))

    # Calculate start and end indices for the middle slice
    # For 120 total frames and 80 target frames: start = (120 - 80) // 2 = 20
    start_idx = (ORIGINAL_SEQUENCE_LENGTH - TARGET_FRAMES) // 2
    end_idx = start_idx + TARGET_FRAMES

    for class_name in class_names:
        class_path = os.path.join(base_path, class_name)

        video_names = sorted([
            d for d in os.listdir(class_path)
            if os.path.isdir(os.path.join(class_path, d))
        ])

        print(
            f"{class_name:<35} "
            f"{len(video_names):>4} videos"
        )

        for video_name in video_names:
            video_path = os.path.join(
                class_path,
                video_name
            )

            frames = []
            valid_video = True

            for frame_num in range(1, ORIGINAL_SEQUENCE_LENGTH + 1):
                frame_file = os.path.join(
                    video_path,
                    f"{frame_num:03d}.npy"
                )

                if not os.path.exists(frame_file):
                    print(
                        f"WARNING: Missing frame: {frame_file}"
                    )
                    valid_video = False
                    break

                frame = np.load(frame_file)

                if frame.shape != (NUM_FEATURES,):
                    print(
                        f"WARNING: Wrong shape: {frame_file} "
                        f"-> {frame.shape}"
                    )
                    valid_video = False
                    break

                frames.append(frame)

            if not valid_video:
                print(
                    f"Skipping incomplete video: "
                    f"{class_name}/{video_name}"
                )
                continue

            full_video_stack = np.stack(frames) # Shape: (120, 130)
            
            # Extract the middle frames
            sliced_video = full_video_stack[start_idx:end_idx] # Shape: (80, 130)

            sequences.append(sliced_video)
            labels.append(class_name)

    X = np.asarray(sequences, dtype=np.float32)
    y = np.asarray(labels)

    print("\nLoaded & Sliced:")
    print("X shape:", X.shape)
    print("y shape:", y.shape)

    return X, y, class_names


# ============================================================
# 4. LOAD TRAIN / VALIDATION / TEST
# ============================================================

start_loading = time.time()

X_train, y_train, train_classes = load_and_slice_sequences_from_folder(
    TRAIN_PATH
)

X_val, y_val, val_classes = load_and_slice_sequences_from_folder(
    VAL_PATH
)

X_test, y_test, test_classes = load_and_slice_sequences_from_folder(
    TEST_PATH
)

print(
    f"\nTotal loading and slicing time: "
    f"{time.time() - start_loading:.2f} seconds"
)


# ============================================================
# 5. BASIC DATA CHECKS
# ============================================================

print("\n" + "=" * 70)
print("DATASET SUMMARY")
print("=" * 70)

print("X_train:", X_train.shape)
print("X_val  :", X_val.shape)
print("X_test :", X_test.shape)

print("Train classes:", len(train_classes))
print("Val classes  :", len(val_classes))
print("Test classes :", len(test_classes))

if X_train.ndim != 3:
    raise ValueError("X_train must be 3-dimensional.")

if X_train.shape[1] != TARGET_FRAMES:
    raise ValueError(
        f"Expected {TARGET_FRAMES} frames after slicing, "
        f"got {X_train.shape[1]}"
    )

if X_train.shape[2] != NUM_FEATURES:
    raise ValueError(
        f"Expected {NUM_FEATURES} features, "
        f"got {X_train.shape[2]}"
    )

if len(train_classes) != EXPECTED_CLASSES:
    raise ValueError(
        f"Expected {EXPECTED_CLASSES} classes, "
        f"found {len(train_classes)}"
    )


# ============================================================
# 6. COMBINE CLASS NAMES
# ============================================================

actions = sorted(
    list(
        set(train_classes)
        | set(val_classes)
        | set(test_classes)
    )
)

print("\nFinal classes:", len(actions))

if len(actions) != EXPECTED_CLASSES:
    raise ValueError(
        f"Expected {EXPECTED_CLASSES} total classes, "
        f"found {len(actions)}"
    )

print("\nClass list:")
for i, action in enumerate(actions):
    print(f"{i:02d}: {action}")


# ============================================================
# 7. LABEL ENCODING
# ============================================================

label_encoder = LabelEncoder()
label_encoder.fit(actions)

y_train_encoded = label_encoder.transform(y_train)
y_val_encoded = label_encoder.transform(y_val)
y_test_encoded = label_encoder.transform(y_test)

print("\nEncoded labels:")
print("Train:", y_train_encoded.shape)
print("Val  :", y_val_encoded.shape)
print("Test :", y_test_encoded.shape)


# ============================================================
# 8. SAVE LABEL ENCODER
# ============================================================

with open(
    os.path.join(RESULTS_DIR, "label_encoder.pkl"),
    "wb"
) as f:
    pickle.dump(label_encoder, f)


# ============================================================
# 9. CLASS DISTRIBUTION
# ============================================================

def save_class_distribution(labels, split_name):
    counts = Counter(labels)
    data = []

    for class_name in actions:
        data.append({
            "class": class_name,
            "count": counts.get(class_name, 0)
        })

    df = pd.DataFrame(data)

    df.to_csv(
        os.path.join(
            RESULTS_DIR,
            f"{split_name}_class_distribution.csv"
        ),
        index=False
    )

    plt.figure(figsize=(16, 7))

    plt.bar(
        range(len(actions)),
        [counts.get(a, 0) for a in actions]
    )

    plt.xticks(
        range(len(actions)),
        actions,
        rotation=90
    )

    plt.xlabel("Class")
    plt.ylabel("Number of Videos")
    plt.title(f"{split_name.capitalize()} Class Distribution (Middle {TARGET_FRAMES} Frames)")

    plt.tight_layout()

    plt.savefig(
        os.path.join(
            RESULTS_DIR,
            f"{split_name}_class_distribution.png"
        ),
        dpi=300
    )

    plt.close()


save_class_distribution(y_train, "train")
save_class_distribution(y_val, "validation")
save_class_distribution(y_test, "test")


# ============================================================
# 10. STANDARDIZATION
# ============================================================

print("\n" + "=" * 70)
print("STANDARDIZATION")
print("=" * 70)

scaler = StandardScaler()

X_train_flat = X_train.reshape(-1, NUM_FEATURES)
X_val_flat = X_val.reshape(-1, NUM_FEATURES)
X_test_flat = X_test.reshape(-1, NUM_FEATURES)

print("Fitting scaler on training data...")

X_train_scaled = scaler.fit_transform(X_train_flat).reshape(X_train.shape)

print("Transforming validation data...")

X_val_scaled = scaler.transform(X_val_flat).reshape(X_val.shape)

print("Transforming test data...")

X_test_scaled = scaler.transform(X_test_flat).reshape(X_test.shape)

with open(
    os.path.join(RESULTS_DIR, "scaler.pkl"),
    "wb"
) as f:
    pickle.dump(scaler, f)

print("Scaling completed.")


# ============================================================
# 11. ONE-HOT ENCODING
# ============================================================

y_train_cat = tf.keras.utils.to_categorical(
    y_train_encoded,
    num_classes=len(actions)
)

y_val_cat = tf.keras.utils.to_categorical(
    y_val_encoded,
    num_classes=len(actions)
)

y_test_cat = tf.keras.utils.to_categorical(
    y_test_encoded,
    num_classes=len(actions)
)


# ============================================================
# 12. BUILD MODEL
# ============================================================

print("\n" + "=" * 70)
print("BUILDING MODEL")
print("=" * 70)

inp = Input(
    shape=(
        TARGET_FRAMES,
        NUM_FEATURES
    )
)

x = Conv1D(
    filters=64,
    kernel_size=3,
    activation="relu",
    padding="same"
)(inp)

x = BatchNormalization()(x)

x = Bidirectional(
    LSTM(
        128,
        return_sequences=True,
        dropout=0.2,
        recurrent_dropout=0.1
    )
)(x)

x = BatchNormalization()(x)
x = Dropout(0.3)(x)

x = Bidirectional(
    LSTM(
        64,
        return_sequences=True,
        dropout=0.2,
        recurrent_dropout=0.1
    )
)(x)

x = BatchNormalization()(x)
x = Dropout(0.3)(x)

query = Dense(64)(x)
value = Dense(64)(x)

x = Attention()([
    query,
    value
])

x = GlobalAveragePooling1D()(x)

x = Dense(
    128,
    activation="relu",
    kernel_regularizer=l2(1e-5)
)(x)

x = BatchNormalization()(x)
x = Dropout(0.3)(x)

x = Dense(
    64,
    activation="relu",
    kernel_regularizer=l2(1e-5)
)(x)

x = BatchNormalization()(x)
x = Dropout(0.3)(x)

out = Dense(
    len(actions),
    activation="softmax"
)(x)

model = Model(
    inputs=inp,
    outputs=out
)


# ============================================================
# 13. COMPILE
# ============================================================

optimizer = Adam(
    learning_rate=LEARNING_RATE
)

model.compile(
    optimizer=optimizer,
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)


# ============================================================
# 14. MODEL SUMMARY
# ============================================================

model.summary()

with open(
    os.path.join(
        RESULTS_DIR,
        "model_summary.txt"
    ),
    "w"
) as f:
    f.reconfigure(encoding="utf-8")
    model.summary(
        print_fn=lambda line: f.write(line + "\n")
    )


# ============================================================
# 15. CALLBACKS
# ============================================================

best_model_path = os.path.join(
    RESULTS_DIR,
    "best_model.keras"
)

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=15,
    restore_best_weights=True,
    verbose=1
)

model_checkpoint = ModelCheckpoint(
    best_model_path,
    monitor="val_accuracy",
    save_best_only=True,
    mode="max",
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=10,
    min_lr=1e-6,
    verbose=1
)


# ============================================================
# 16. TRAINING
# ============================================================

print("\n" + "=" * 70)
print("TRAINING")
print("=" * 70)

training_start = time.time()

history = model.fit(
    X_train_scaled,
    y_train_cat,
    validation_data=(
        X_val_scaled,
        y_val_cat
    ),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[
        early_stopping,
        model_checkpoint,
        reduce_lr
    ],
    verbose=1
)

training_time = time.time() - training_start

print(
    f"\nTraining completed in "
    f"{training_time / 60:.2f} minutes"
)


# ============================================================
# 17. SAVE TRAINING HISTORY
# ============================================================

history_df = pd.DataFrame(
    history.history
)

history_df.index.name = "epoch"

history_df.to_csv(
    os.path.join(
        RESULTS_DIR,
        "training_history.csv"
    )
)


# ============================================================
# 18. TRAINING ACCURACY PLOT
# ============================================================

plt.figure(figsize=(10, 6))

plt.plot(
    history.history["accuracy"],
    label="Training Accuracy"
)

plt.plot(
    history.history["val_accuracy"],
    label="Validation Accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title(
    f"Training and Validation Accuracy (Middle {TARGET_FRAMES} Frames)"
)

plt.legend()
plt.grid(True)
plt.tight_layout()

plt.savefig(
    os.path.join(
        RESULTS_DIR,
        "training_validation_accuracy.png"
    ),
    dpi=300
)

plt.close()


# ============================================================
# 19. TRAINING LOSS PLOT
# ============================================================

plt.figure(figsize=(10, 6))

plt.plot(
    history.history["loss"],
    label="Training Loss"
)

plt.plot(
    history.history["val_loss"],
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title(
    f"Training and Validation Loss (Middle {TARGET_FRAMES} Frames)"
)

plt.legend()
plt.grid(True)
plt.tight_layout()

plt.savefig(
    os.path.join(
        RESULTS_DIR,
        "training_validation_loss.png"
    ),
    dpi=300
)

plt.close()


# ============================================================
# 20. LOAD BEST MODEL
# ============================================================

print("\nLoading best saved model...")

best_model = load_model(
    best_model_path
)


# ============================================================
# 21. TEST EVALUATION
# ============================================================

print("\n" + "=" * 70)
print("TEST EVALUATION")
print("=" * 70)

test_loss, test_accuracy = best_model.evaluate(
    X_test_scaled,
    y_test_cat,
    batch_size=BATCH_SIZE,
    verbose=1
)

print(f"\nTest Loss     : {test_loss:.6f}")
print(f"Test Accuracy : {test_accuracy:.6f}")


# ============================================================
# 22. PREDICTIONS
# ============================================================

y_prob = best_model.predict(
    X_test_scaled,
    batch_size=BATCH_SIZE,
    verbose=1
)

y_pred = np.argmax(
    y_prob,
    axis=1
)


# ============================================================
# 23. BEST EPOCH
# ============================================================

best_epoch = (
    np.argmax(
        history.history["val_accuracy"]
    ) + 1
)

best_val_accuracy = max(
    history.history["val_accuracy"]
)

best_val_loss = min(
    history.history["val_loss"]
)


# ============================================================
# 24. OVERALL RESULTS
# ============================================================

overall_results = pd.DataFrame([{

    "Experiment":
        f"130 Features - Middle {TARGET_FRAMES} Frames",

    "Sequence Length":
        TARGET_FRAMES,

    "Number of Features":
        NUM_FEATURES,

    "Number of Classes":
        len(actions),

    "Train Samples":
        len(X_train),

    "Validation Samples":
        len(X_val),

    "Test Samples":
        len(X_test),

    "Test Loss":
        test_loss,

    "Test Accuracy":
        test_accuracy,

    "Best Epoch":
        best_epoch,

    "Best Validation Accuracy":
        best_val_accuracy,

    "Best Validation Loss":
        best_val_loss,

    "Training Time Seconds":
        training_time

}])

overall_results.to_csv(
    os.path.join(
        RESULTS_DIR,
        "overall_metrics.csv"
    ),
    index=False
)


# ============================================================
# 25. CLASSIFICATION REPORT
# ============================================================

report_dict = classification_report(
    y_test_encoded,
    y_pred,
    target_names=actions,
    output_dict=True,
    zero_division=0
)

report_text = classification_report(
    y_test_encoded,
    y_pred,
    target_names=actions,
    zero_division=0
)

print("\n" + "=" * 70)
print("CLASSIFICATION REPORT")
print("=" * 70)

print(report_text)

with open(
    os.path.join(
        RESULTS_DIR,
        "classification_report.txt"
    ),
    "w"
) as f:
    f.write(report_text)

report_df = pd.DataFrame(
    report_dict
).transpose()

report_df.to_csv(
    os.path.join(
        RESULTS_DIR,
        "classification_report.csv"
    )
)


# ============================================================
# 26. PER-CLASS METRICS
# ============================================================

per_class_metrics = pd.DataFrame({
    "class": actions,
    "precision": precision_recall_fscore_support(
        y_test_encoded,
        y_pred,
        labels=range(len(actions)),
        zero_division=0
    )[0],
    "recall": precision_recall_fscore_support(
        y_test_encoded,
        y_pred,
        labels=range(len(actions)),
        zero_division=0
    )[1],
    "f1_score": precision_recall_fscore_support(
        y_test_encoded,
        y_pred,
        labels=range(len(actions)),
        zero_division=0
    )[2],
    "support": precision_recall_fscore_support(
        y_test_encoded,
        y_pred,
        labels=range(len(actions)),
        zero_division=0
    )[3]
})

per_class_metrics.to_csv(
    os.path.join(
        RESULTS_DIR,
        "per_class_metrics.csv"
    ),
    index=False
)


# ============================================================
# 27. CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    y_test_encoded,
    y_pred,
    labels=range(len(actions))
)

cm_df = pd.DataFrame(
    cm,
    index=actions,
    columns=actions
)

cm_df.to_csv(
    os.path.join(
        RESULTS_DIR,
        "confusion_matrix.csv"
    )
)


# ============================================================
# 28. CONFUSION MATRIX FIGURE
# ============================================================

plt.figure(
    figsize=(22, 20)
)

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=actions,
    yticklabels=actions,
    cbar=True
)

plt.xlabel("Predicted Class")
plt.ylabel("True Class")
plt.title(
    f"Confusion Matrix — 130 Features × Middle {TARGET_FRAMES} Frames"
)

plt.xticks(rotation=90, fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()

plt.savefig(
    os.path.join(
        RESULTS_DIR,
        "confusion_matrix.png"
    ),
    dpi=300
)

plt.close()


# ============================================================
# 29. NORMALIZED CONFUSION MATRIX
# ============================================================

cm_normalized = (
    cm.astype(np.float64)
    /
    cm.sum(
        axis=1,
        keepdims=True
    )
)

cm_normalized = np.nan_to_num(
    cm_normalized
)

cm_normalized_df = pd.DataFrame(
    cm_normalized,
    index=actions,
    columns=actions
)

cm_normalized_df.to_csv(
    os.path.join(
        RESULTS_DIR,
        "normalized_confusion_matrix.csv"
    )
)

plt.figure(
    figsize=(22, 20)
)

sns.heatmap(
    cm_normalized,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    xticklabels=actions,
    yticklabels=actions,
    cbar=True
)

plt.xlabel("Predicted Class")
plt.ylabel("True Class")
plt.title(
    f"Normalized Confusion Matrix — 130 Features × Middle {TARGET_FRAMES} Frames"
)

plt.xticks(rotation=90, fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()

plt.savefig(
    os.path.join(
        RESULTS_DIR,
        "normalized_confusion_matrix.png"
    ),
    dpi=300
)

plt.close()


# ============================================================
# 30. SAVE TEST PREDICTIONS
# ============================================================

prediction_data = {
    "true_label": [
        actions[i]
        for i in y_test_encoded
    ],
    "predicted_label": [
        actions[i]
        for i in y_pred
    ],
    "confidence": np.max(
        y_prob,
        axis=1
    )
}

predictions_df = pd.DataFrame(
    prediction_data
)

predictions_df.to_csv(
    os.path.join(
        RESULTS_DIR,
        "test_predictions.csv"
    ),
    index=False
)


# ============================================================
# 31. FINAL SUMMARY
# ============================================================

summary_text = f"""
============================================================
STAGE 2 FINAL SUMMARY
============================================================

Experiment:
130 Features × Middle {TARGET_FRAMES} Frames (Extracted from 120)

Dataset:
Student-Independent

Number of Classes:
{len(actions)}

Training Samples:
{len(X_train)}

Validation Samples:
{len(X_val)}

Testing Samples:
{len(X_test)}

------------------------------------------------------------
MODEL
------------------------------------------------------------

Conv1D:
64 filters, kernel size 3

BiLSTM:
128 units

BiLSTM:
64 units

Attention:
Dense query/value = 64

GlobalAveragePooling1D

Dense:
128

Dense:
64

Output:
{len(actions)} classes

Optimizer:
Adam

Learning Rate:
{LEARNING_RATE}

Batch Size:
{BATCH_SIZE}

Maximum Epochs:
{EPOCHS}

------------------------------------------------------------
RESULTS
------------------------------------------------------------

Test Loss:
{test_loss:.6f}

Test Accuracy:
{test_accuracy:.6f}

------------------------------------------------------------
TRAINING
------------------------------------------------------------

Best Epoch:
{best_epoch}

Best Validation Accuracy:
{best_val_accuracy:.6f}

Best Validation Loss:
{best_val_loss:.6f}

Training Time:
{training_time / 60:.2f} minutes

============================================================
"""

print(summary_text)

with open(
    os.path.join(
        RESULTS_DIR,
        "final_summary.txt"
    ),
    "w"
) as f:
    f.write(summary_text)


# ============================================================
# 32. FINAL MESSAGE
# ============================================================

print("\n" + "=" * 70)
print("STAGE 2 COMPLETED")
print("=" * 70)

print(f"\nExperiment:")
print(f"130 Features × Middle {TARGET_FRAMES} Frames")

print("\nResults saved to:")
print(RESULTS_DIR)

print("\nTest Accuracy:")
print(f"{test_accuracy:.4f}")

print("\nBest Validation Accuracy:")
print(f"{best_val_accuracy:.4f}")

print("\nDone.")

CHECKING DATASET
Train path: D:\aaa EAAI Major Revision\dynamic\dynamic_dataset\csv_dataset\train_csv
Validation path: D:\aaa EAAI Major Revision\dynamic\dynamic_dataset\csv_dataset\validation_csv
Test path: D:\aaa EAAI Major Revision\dynamic\dynamic_dataset\csv_dataset\test_csv

Loading and slicing: D:\aaa EAAI Major Revision\dynamic\dynamic_dataset\csv_dataset\train_csv
Classes found: 40
Good to See You                      174 videos
Leave                                150 videos
Please                               159 videos
What Time is It                      170 videos
Where Do You Live                    175 videos
abar dekha hbe                       162 videos
ami dukkhito                         161 videos
ami valo achi                        156 videos
apnake amar valo legeche             166 videos
apnake kivabe sahajjo korte pari     156 videos
apnar nam ki                         150 videos
apni kemon achen                     155 videos
apni ki kaj koren              

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 60, 130)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 60, 64)    │     25,024 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 60, 64)    │        256 │ conv1d[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ (None, 60, 256)   │    197,632 │ batch_normalizat… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 60, 256)   │      1,024 │ bidirectional[0]… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 60, 256)   │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_1     │ (None, 60, 128)   │    164,352 │ dropout[0][0]     │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 60, 128)   │        512 │ bidirectional_1[… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 60, 128)   │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 60, 64)    │      8,256 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 60, 64)    │      8,256 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention           │ (None, 60, 64)    │          0 │ dense[0][0],      │
│ (Attention)         │                   │            │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 64)        │          0 │ attention[0][0]   │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 128)       │      8,320 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128)       │        512 │ dense_2[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 128)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 64)        │      8,256 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64)        │        256 │ dense_3[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 64)        │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 40)        │      2,600 │ dropout_3[0][0] 

 Total params: 425,256 (1.62 MB)

 Trainable params: 423,976 (1.62 MB)

 Non-trainable params: 1,280 (5.00 KB)


TRAINING
Epoch 1/200
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 650ms/step - accuracy: 0.0642 - loss: 3.9241
Epoch 1: val_accuracy improved from None to 0.18588, saving model to D:\aaa EAAI Major Revision\dynamic\dynamic_dataset\results_130features_middle80frames\best_model.keras
97/97 ━━━━━━━━━━━━━━━━━━━━ 91s 702ms/step - accuracy: 0.0932 - loss: 3.5939 - val_accuracy: 0.1859 - val_loss: 3.1216 - learning_rate: 5.0000e-04
Epoch 2/200
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 653ms/step - accuracy: 0.2048 - loss: 2.8745
Epoch 2: val_accuracy improved from 0.18588 to 0.51922, saving model to D:\aaa EAAI Major Revision\dynamic\dynamic_dataset\results_130features_middle80frames\best_model.keras
97/97 ━━━━━━━━━━━━━━━━━━━━ 65s 671ms/step - accuracy: 0.2397 - loss: 2.7229 - val_accuracy: 0.5192 - val_loss: 2.0921 - learning_rate: 5.0000e-04
Epoch 3/200
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 640ms/step - accuracy: 0.3535 - loss: 2.2636
Epoch 3: val_accuracy improved from 0.51922 to 0.63608, saving model to D:\aaa EAAI Major 

In [1]:
# ============================================================
# FINAL BULLETPROOF SCRIPT: Student-Independent Sign Language
# ============================================================

import os
import time
import pickle
import json
import numpy as np
import pandas as pd

from collections import Counter
from sklearn.preprocessing import StandardScaler, LabelEncoder

import tensorflow as tf
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import (
    Input,
    Conv1D,
    LSTM,
    Dense,
    Dropout,
    BatchNormalization,
    Bidirectional,
    Attention,
    GlobalAveragePooling1D
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau
)

# ============================================================
# 1. CONFIGURATION & PATHS
# ============================================================

BASE_DIR = r"D:\aaa EAAI Major Revision\dynamic\dynamic_dataset"
DATA_DIR = os.path.join(BASE_DIR, "csv_dataset")

TRAIN_PATH = os.path.join(DATA_DIR, "train_csv")
VAL_PATH = os.path.join(DATA_DIR, "validation_csv")
TEST_PATH = os.path.join(DATA_DIR, "test_csv")

RESULTS_DIR = os.path.join(BASE_DIR, "results_130features_60frames")
os.makedirs(RESULTS_DIR, exist_ok=True)

SEQUENCE_LENGTH = 60
NUM_FEATURES = 130
EXPECTED_CLASSES = 40

BATCH_SIZE = 64
EPOCHS = 200
LEARNING_RATE = 0.0005

np.random.seed(42)
tf.random.set_seed(42)

# ============================================================
# 2. SAFE DATA LOADING PIPELINE
# ============================================================

def load_sequences_safely(base_path):
    if not os.path.exists(base_path):
        raise FileNotFoundError(f"Directory missing: {base_path}")
        
    class_names = sorted([
        d for d in os.listdir(base_path)
        if os.path.isdir(os.path.join(base_path, d))
    ])

    sequences = []
    labels = []

    print(f"\nScanning: {base_path} ({len(class_names)} classes found)")

    for class_name in class_names:
        class_path = os.path.join(base_path, class_name)
        video_names = sorted([
            d for d in os.listdir(class_path)
            if os.path.isdir(os.path.join(class_path, d))
        ])

        for video_name in video_names:
            video_path = os.path.join(class_path, video_name)
            frames = []
            valid = True

            for frame_num in range(1, SEQUENCE_LENGTH + 1):
                frame_file = os.path.join(video_path, f"{frame_num:03d}.npy")
                
                if not os.path.exists(frame_file):
                    valid = False
                    break
                
                try:
                    frame = np.load(frame_file).astype(np.float32)
                except Exception:
                    valid = False
                    break

                if frame.shape != (NUM_FEATURES,):
                    valid = False
                    break

                frames.append(frame)

            if valid and len(frames) == SEQUENCE_LENGTH:
                sequences.append(np.stack(frames))
                labels.append(class_name)

    return np.array(sequences, dtype=np.float32), np.array(labels), class_names

# Load all splits
X_train, y_train, train_classes = load_sequences_safely(TRAIN_PATH)
X_val, y_val, val_classes = load_sequences_safely(VAL_PATH)
X_test, y_test, test_classes = load_sequences_safely(TEST_PATH)

print(f"\nShapes Loaded -> Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

# ============================================================
# 3. ENCODING & SCALING
# ============================================================

actions = sorted(list(set(train_classes) | set(val_classes) | set(test_classes)))
if len(actions) != EXPECTED_CLASSES:
    print(f"Warning: Found {len(actions)} classes, expected {EXPECTED_CLASSES}.")

label_encoder = LabelEncoder()
label_encoder.fit(actions)

y_train_enc = label_encoder.transform(y_train)
y_val_enc = label_encoder.transform(y_val)
y_test_enc = label_encoder.transform(y_test)

# Save encoder artifacts
with open(os.path.join(RESULTS_DIR, "label_encoder.pkl"), "wb") as f:
    pickle.dump(label_encoder, f)

# Flatten for Scaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train.reshape(-1, NUM_FEATURES)).reshape(X_train.shape)
X_val_scaled = scaler.transform(X_val.reshape(-1, NUM_FEATURES)).reshape(X_val.shape)
X_test_scaled = scaler.transform(X_test.reshape(-1, NUM_FEATURES)).reshape(X_test.shape)

with open(os.path.join(RESULTS_DIR, "scaler.pkl"), "wb") as f:
    pickle.dump(scaler, f)

y_train_cat = tf.keras.utils.to_categorical(y_train_enc, num_classes=len(actions))
y_val_cat = tf.keras.utils.to_categorical(y_val_enc, num_classes=len(actions))
y_test_cat = tf.keras.utils.to_categorical(y_test_enc, num_classes=len(actions))

# ============================================================
# 4. MODEL ARCHITECTURE (CONV1D + BiLSTM + ATTENTION)
# ============================================================

inp = Input(shape=(SEQUENCE_LENGTH, NUM_FEATURES))
x = Conv1D(64, kernel_size=3, activation="relu", padding="same")(inp)
x = BatchNormalization()(x)

x = Bidirectional(LSTM(128, return_sequences=True, dropout=0.2))(x)
x = BatchNormalization()(x)
x = Dropout(0.3)(x)

x = Bidirectional(LSTM(64, return_sequences=True, dropout=0.2))(x)
x = BatchNormalization()(x)
x = Dropout(0.3)(x)

query = Dense(64)(x)
value = Dense(64)(x)
x = Attention()([query, value])

x = GlobalAveragePooling1D()(x)
x = Dense(128, activation="relu", kernel_regularizer=l2(1e-5))(x)
x = BatchNormalization()(x)
x = Dropout(0.3)(x)

out = Dense(len(actions), activation="softmax")(x)
model = Model(inputs=inp, outputs=out)

model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

# ============================================================
# 5. TRAINING EXECUTION
# ============================================================

best_model_path = os.path.join(RESULTS_DIR, "best_model.keras")

callbacks = [
    EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True, verbose=1),
    ModelCheckpoint(best_model_path, monitor="val_accuracy", save_best_only=True, mode="max", verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=10, min_lr=1e-6, verbose=1)
]

history = model.fit(
    X_train_scaled, y_train_cat,
    validation_data=(X_val_scaled, y_val_cat),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)

# Save metadata for UI / Deployment
with open(os.path.join(RESULTS_DIR, "classes.json"), "w", encoding="utf-8") as f:
    json.dump(actions, f, indent=4)

print("\nTraining completed successfully! Model and checkpoints saved.")


Scanning: D:\aaa EAAI Major Revision\dynamic\dynamic_dataset\csv_dataset\train_csv (40 classes found)

Scanning: D:\aaa EAAI Major Revision\dynamic\dynamic_dataset\csv_dataset\validation_csv (40 classes found)

Scanning: D:\aaa EAAI Major Revision\dynamic\dynamic_dataset\csv_dataset\test_csv (40 classes found)

Shapes Loaded -> Train: (6179, 60, 130), Val: (1275, 60, 130), Test: (1212, 60, 130)
Epoch 1/200
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 189ms/step - accuracy: 0.1104 - loss: 3.5491
Epoch 1: val_accuracy improved from None to 0.21961, saving model to D:\aaa EAAI Major Revision\dynamic\dynamic_dataset\results_130features_60frames\best_model.keras

Epoch 1: finished saving model to D:\aaa EAAI Major Revision\dynamic\dynamic_dataset\results_130features_60frames\best_model.keras
97/97 ━━━━━━━━━━━━━━━━━━━━ 38s 223ms/step - accuracy: 0.1104 - loss: 3.5491 - val_accuracy: 0.2196 - val_loss: 3.1129 - learning_rate: 5.0000e-04
Epoch 2/200
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 197ms/step - accuracy: 0.283